# STEP00. 눈 트랙 전체 개요

## 역할

눈 데이터 전처리, 모델 학습, 영상 평가 순서와 산출물을 정리한다. 이 노트북에서는 계산하지 않는다.

## 데이터

| 데이터 | 규모 | 사용 목적 |
|---|---:|---|
| MRL Eye | 84,898장 / 37명 | 눈 CNN 학습 |
| DMD | 16영상 / 13명 | 눈 crop 추가 학습·평가 |
| Yawn-Eye | 클래스당 600장 | 기존 01·02 모델용 |
| NITYMED | microsleep 19 / 하품 107 | 두 영상 집단의 눈 지표 비교 |

## 노트북 구성

| 번호 | 역할 |
|---|---|
| 00 | 전체 순서와 산출물 확인 |
| 10 | DMD annotation → 프레임 눈 상태 GT |
| 11 | MRL·DMD 눈 데이터와 manifest 생성 |
| 12 | 눈 CNN 학습 조건 비교 |
| 13 | DMD 원본 영상 파이프라인 평가 |
| 14 | NITYMED 영상 단위 눈 지표 비교 |
| 15 | 하품 구간의 Closed 판정 분석 |
| 20·21 | 하품 파트(팀원) |
| 30 | 눈·하품 지표 통합 |


## 실행 순서

```text
10  DMD annotation → 프레임 GT
 ↓
11  DMD 눈 crop + MRL·DMD 통합 manifest
 ↓
12  눈 CNN 학습 A/B/C/C′ 비교
 ↓
13  DMD 영상 평가   14  NITYMED 비교   15  하품 구간 분석
```

STEP13~15는 STEP12의 C 모델과 임계값 0.93을 사용한다.

## 주요 모듈

| 모듈 | 역할 |
|---|---|
| `dmd_annotation.py` | DMD annotation 읽기 |
| `build_dmd_eye_dataset.py` | DMD 눈 crop 생성·경로 처리 |
| `mrl_split.py` | MRL subject 분할 |
| `mrl_dataset.py` · `eye_preprocess.py` | manifest와 공통 전처리 |
| `train_eye.py` | 눈 CNN 학습·평가 |


In [ ]:
# [셀 1] 환경·산출물 점검 (지도 노트북이라 계산은 최소)

import sys
from pathlib import Path

_anchors = []
if "__vsc_ipynb_file__" in globals():
    _anchors.append(Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
if globals().get("_dh"):
    _anchors.append(Path(globals()["_dh"][0]).resolve())
_anchors.append(Path.cwd().resolve())

_root = next(
    (p for a in _anchors for p in [a, *a.parents]
     if (p / "config.py").is_file() and (p / "requirements.txt").is_file()),
    None,
)
if _root is not None:
    if str(_root) in sys.path:
        sys.path.remove(str(_root))
    sys.path.insert(0, str(_root))

import config

print("PROJECT_ROOT :", config.PROJECT_ROOT.name)
print()

# 각 STEP 의 핵심 산출물이 있는지만 확인한다. 없으면 그 STEP 을 아직 안 돌린 것.
checks = [
    ("STEP10 DMD 프레임 GT",   config.OUTPUTS_DIR / "dmd_gt" / "_summary.csv"),
    ("STEP11 통합 manifest",   config.OUTPUTS_DIR / "eye_dataset" / "eye_manifest.csv"),
    ("STEP12 조건 비교",       config.OUTPUTS_DIR / "eye_dataset" / "step12_comparison.csv"),
    ("STEP12 C 모델",          config.ARTIFACT_DIR / "eye_mrl+dmd__eval-dmd__gray128.keras"),
    ("STEP13 프레임 평가",     config.OUTPUTS_DIR / "eye_frame_eval" / "frame_eval_summary.csv"),
    ("STEP14 NITYMED 영상평가", config.OUTPUTS_DIR / "nitymed_eval" / "video_metrics.csv"),
    ("STEP15 하품 오경보",     config.OUTPUTS_DIR / "yawn_fp" / "video_closed_rate.csv"),
    ("YuNet 검출기",           config.YUNET_MODEL),
]
for label, p in checks:
    print(f"  [{'o' if Path(p).exists() else 'X'}] {label:24s} {config._rel(p)}")

### 관찰 결과

- 각 STEP 의 핵심 산출물이 있으면 `[o]`, 없으면 `[X]` 다. `[X]` 인 STEP 은 해당 노트북을 아직 실행하지 않은 것이다.
- 이 노트북은 산출물을 만들지 않는다. 점검만 한다.

## 결과 요약

### 데이터 구축

- DMD 16영상과 annotation의 프레임 정렬 확인: **16/16**
- 이진 눈 상태 사용 가능 프레임: **65,302개(74.0%)**
- 통합 manifest: **106,802행**, 누수 검사 6개 항목 모두 0
- MRL+DMD train의 Closed 비율: **약 49.6%**

### DMD hold-out 결과

| 조건 | Closed-Recall | Precision | F1 |
|---|---:|---:|---:|
| **C (MRL+DMD)** | **0.914** | 0.787 | **0.846** |
| C′ (MRL→DMD) | 0.881 | 0.806 | 0.842 |
| A (MRL only) | 0.853 | 0.317 | 0.462 |
| B (DMD only) | 0.597 | 0.955 | 0.735 |

이번 한 번의 학습에서는 C의 Recall과 F1이 가장 높았다. 단일 seed와 한 개의 test 분할 결과이므로 일반화하지 않는다.

### 영상 평가

- STEP13: DMD 원본 영상 파이프라인 Recall **0.916**, F1 **0.853**
- STEP14: 창 개수 보정 후 눈 지표 AUC **0.402~0.424**
- STEP15: 하품 구간 Closed **25.7%**, microsleep 영상 표본 **19.9%**

NITYMED 결과는 정상 운전과 졸음을 비교한 성능이 아니다. microsleep 영상과 하품 영상의 눈 지표를 비교한 결과다.

## 한계와 다음 작업

- DMD는 13명·16영상이며 모두 `Car Stopped` 조건이다.
- NITYMED 두 집단은 영상 길이가 다르고 인물 ID를 확인하기 어렵다.
- STEP12는 seed 42로 한 번씩 학습했다.
- 다음 단계는 하품 결과와 시간축을 맞춰 통합 지표를 만드는 것이다.
